# Air Quality Data Analysis with Python
## Notebook 3 · pandas and a Year of Real Data

⏱️ About 60 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebooks 1–2

In Notebook 2 you handled one day of measurements with a list. Real analysis means
thousands of hours. **pandas** is Python's table library — it wraps the
list-of-records idea in an object called a `DataFrame` that can load, filter,
summarise and resample data in a line or two.

You'll work with a real dataset: **12 months of hourly PM2.5 from Oshodi Bus
Terminal, Lagos** (July 2025 – June 2026, measured by an AirGradient low-cost
sensor in the OpenAQ network). You'll learn to:

* load a CSV file into a `DataFrame`,
* turn text timestamps into real datetimes (and handle **time zones**),
* summarise with `.describe()` and boolean tricks,
* select time ranges and resample hourly data to daily means,
* clean a **raw sensor file** the way it really arrives,
* and check a dataset's **completeness** before trusting it.

### 1. Loading the data

First, the standard import — nearly every data analysis in Python starts with this
line (`pd` is the universal nickname):

In [ ]:
import pandas as pd

The next cell points at the course data. It works in both places you might be
running: in Colab it reads straight from the course repository on GitHub; if you
downloaded the repository to your own computer, it reads the local files.

In [ ]:
from pathlib import Path

DATA = "../data"  # local checkout of the course repository
if not Path(DATA).exists():  # running in Colab -> read from GitHub
    DATA = "https://raw.githubusercontent.com/rwpinder/tutorial-air-quality-data-analysis/main/data"

`pd.read_csv(...)` loads a CSV file into a `DataFrame`. `.head()` shows the first
five rows — always look at your data before computing anything:

In [ ]:
lagos = pd.read_csv(f"{DATA}/lagos_pm25_recent.csv")
lagos.head()

Two columns: `datetime` (when) and `pm25_value` (µg/m³). Each row is one hour —
exactly the records shape from Notebook 2, just 7,000+ rows instead of 3.

In [ ]:
print(f"Rows: {len(lagos)}")
print(f"Columns: {list(lagos.columns)}")

### 2. Making time real: datetimes and time zones

`.info()` reveals a problem: pandas loaded `datetime` as `object` — plain text.
Text can't answer "what month is this?" or "give me January only".

In [ ]:
lagos.info()

The fix is a recipe you will reuse in every notebook from now on. Three steps:

1. **parse** the text into real datetimes (`utc=True` reads them as Coordinated
   Universal Time, which is how the data is stored),
2. **index** the table by time (`set_index`), unlocking time-based selection,
3. **convert** to local clock time — Lagos uses West Africa Time (WAT), one hour
   ahead of UTC all year.

Step 3 matters more than it looks: rush hour happens on the *local* clock. Skip it
and every hour-of-day result in Notebook 5 would be shifted by one hour.

In [ ]:
lagos["datetime"] = pd.to_datetime(lagos["datetime"], utc=True)
lagos = lagos.set_index("datetime")
lagos = lagos.tz_convert("Africa/Lagos")
lagos.head()

Note the `+01:00` in the index now — timestamps are on Lagos time.

### 3. First summary

`.describe()` summarises a whole column in one call:

In [ ]:
lagos["pm25_value"].describe()

Read those numbers like an analyst:

* **mean ≈ 44 µg/m³** — nearly **9×** the WHO *annual* guideline of 5 µg/m³.
* **max ≈ 283 µg/m³** — hazardous by any standard.
* The **50%** row is the median (~41): half of all hours were above it.

Run this cell to define the exercise checker (same as before):

In [ ]:
def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")

### 4. Boolean questions at scale

In Notebook 2 you *looped* to count hours above a threshold. pandas does it with a
comparison on the whole column. And there's a lovely trick: `True` counts as 1 and
`False` as 0, so **the mean of a comparison is the fraction of rows where it's
true**:

In [ ]:
pm = lagos["pm25_value"]  # a single column is called a Series

fraction_above = (pm > 15).mean()
print(f"Fraction of hours above the WHO 24-h guideline level: {fraction_above:.1%}")

**98.8%.** At this location, an hour of air meeting the WHO guideline level is a
rare event — that single number is a finding you could put in a report title.

✏️ **Your turn 3.1** — What fraction of hours were at or above **100 µg/m³** (a
level where everyone should reduce outdoor exertion)? Store it in `fraction_severe`.

In [ ]:
fraction_severe = ...
print("Fraction of hours at or above 100 µg/m³:", fraction_severe)

In [ ]:
check("fraction_severe", lambda: 0.01 < fraction_severe < 0.03,
      "compare pm >= 100, then take .mean() of the comparison")

### 5. From hours to days: resample

Health guidelines talk about **24-hour averages**, so hourly data usually needs to
become daily data. With a datetime index this is one line — `.resample("D")` groups
the rows into calendar days, and `.mean()` averages each group:

In [ ]:
daily = pm.resample("D").mean()
daily.head()

`resample` creates a row for **every** calendar day — including days the sensor
didn't report, which appear as `NaN` ("not a number", pandas' marker for missing).
Count them before you trust any daily statistic:

In [ ]:
print(f"Days in the record:  {len(daily)}")
print(f"Days with no data:   {daily.isna().sum()}")

45 missing days ≈ 13% of the year. Low-cost sensors lose power, lose connectivity,
get moved. Knowing *how much* is missing (and whether the gaps cluster in one
season) is part of every honest analysis.

✏️ **Your turn 3.2** — Which day had the worst air? Find the highest daily mean
(`worst_value`) and the day it happened (`worst_day`). Two useful methods:
`.max()` gives a Series' largest value, `.idxmax()` gives the index label (here:
the day) where it occurs.

In [ ]:
worst_value = ...
worst_day = ...
print("Worst day:", worst_day, "with a daily mean of", worst_value, "µg/m³")

In [ ]:
check("worst_value", lambda: 90 < worst_value < 100,
      "daily.max() — the highest daily mean is in the mid-90s")
check("worst_day is in Harmattan", lambda: worst_day.month == 1,
      "daily.idxmax() returns the day; it falls in January, peak Harmattan")

### 6. Selecting time ranges

A datetime index lets you select by label with `.loc` — a whole month with
`"2026-01"`, or any range with `"start":"end"`:

In [ ]:
january = pm.loc["2026-01"]
print(f"January 2026 mean: {january.mean():.1f} µg/m³ over {len(january)} hours")

✏️ **Your turn 3.3** — Compute `april_mean`, the mean for April 2026. How does the
heart of the rainy build-up compare with Harmattan January?

In [ ]:
april_mean = ...
print("April 2026 mean:", april_mean, "µg/m³")

In [ ]:
check("april_mean", lambda: 34 < april_mean < 41,
      'select the month with pm.loc["2026-04"], then .mean()')

January runs roughly **60% higher** than April. You'll turn that into the monthly
chart — the second signature plot of this course — in Notebook 5.

### 7. Raw data, as it really arrives

The file you've used so far was prepared for teaching. Real sensor exports are
messier. `oshodi_raw_january.csv` is one month from the same sensor **exactly as
archived by OpenAQ** — have a look:

In [ ]:
raw = pd.read_csv(f"{DATA}/oshodi_raw_january.csv")
raw.head()

The `parameter` column is the tell: this file mixes **five different measurements**
in one table. `.value_counts()` shows what's there:

In [ ]:
raw["parameter"].value_counts()

PM2.5, PM1, relative humidity, temperature, and `um003` (a particle count). Using
`value` without filtering would average temperatures with concentrations —
nonsense. Filter with a boolean mask, exactly like section 4 (`.copy()` tells
pandas we want an independent table, not a view of the original):

In [ ]:
raw_pm25 = raw[raw["parameter"] == "pm25"].copy()
print(f"{len(raw_pm25)} PM2.5 rows out of {len(raw)} total")

✏️ **Your turn 3.4** — Apply the three-step datetime recipe from section 2 to
`raw_pm25`, ending with a Series `raw_series` of PM2.5 values on Lagos time.
(Parse `raw_pm25["datetime"]`, set it as the index, `tz_convert`, then take the
`"value"` column. One extra wrinkle: `value` also loaded as text — convert it with
`pd.to_numeric(...)`.)

In [ ]:
# 1. parse raw_pm25["datetime"]   2. set_index + tz_convert   3. to_numeric on "value"
raw_series = ...

In [ ]:
check("raw_series length", lambda: 650 < len(raw_series) < 750,
      "after filtering to pm25 there are about 700 hourly rows in January")
check("raw_series is on Lagos time", lambda: str(raw_series.index.tz) == "Africa/Lagos",
      "parse with utc=True, set_index, then .tz_convert('Africa/Lagos')")
check("raw January mean", lambda: 55 < raw_series.mean() < 68,
      "if the mean looks wrong, make sure you filtered parameter == 'pm25' first")

Your cleaned January mean should match section 6's `january.mean()` — same sensor,
same month, two very different starting files. **That agreement is the point of
cleaning.**

### 8. The gap check: why we look before we trust

One more real file. Lagos *does* have a reference-grade monitor (at the US
Consulate) — so why does this course use a low-cost sensor for Lagos? Load the
consulate record and check the spacing between consecutive measurements:

In [ ]:
consulate = pd.read_csv(f"{DATA}/lagos_us_consulate_2023_2024.csv")
consulate["datetime"] = pd.to_datetime(consulate["datetime"], utc=True)
consulate = consulate.set_index("datetime").tz_convert("Africa/Lagos")

step = consulate.index.to_series().diff()
print(f"Longest gap between measurements: {step.max()}")
print(f"Gaps longer than a week: {(step > pd.Timedelta('7D')).sum()}")

A **203-day** outage — the monitor was down for most of 2024. A year-long analysis
from this record would silently describe only the seasons it was awake for. The
lesson generalises:

> **Before analysing any dataset: how complete is it, and where are the gaps?**

(The AQ agent runs automated quality-control checks like these — flagging
suspicious zeros, gaps, and drift — on every dataset it ingests.)

### 9. Recap

* `pd.read_csv` loads data; **look first** with `.head()` and `.info()`.
* The datetime recipe: `pd.to_datetime(..., utc=True)` → `set_index` →
  `tz_convert("Africa/Lagos")`. Local time is not optional.
* `.describe()`, and the boolean-mean trick for "fraction of hours above X".
* `.resample("D").mean()` for daily averages — then **count the NaNs**.
* `.loc["2026-01"]` selects time ranges by label.
* Raw files need cleaning: filter the parameter, coerce numbers, check gaps.

**Next: Notebook 4 — First Plots**, where this year of data becomes pictures.